# Classification and Regression Trees (CART)

Notebook created by Eloise Berson

In [1]:
# load numpy as np, matplotlib as plt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn import model_selection
from sklearn import tree
from sklearn import ensemble
%matplotlib inline

In [2]:
plt.rc('font', **{'size': 12}) # police size

# 1. Loading the dataset


## Goal
The goal of this notebook is to build a model that predicts the **presence of heart disease in a patient**.  

We will work with the [UCI Heart Disease dataset](https://archive.ics.uci.edu/dataset/45/heart+disease), slightly modified.  

The dataset includes a header row describing the features, followed by one row per patient. Each row contains demographic, clinical, and diagnostic information.  

---

## Features

### Demographic Information
- **age**: patient’s age  
- **sex**: sex (1 = male, 0 = female)  

### Clinical and Diagnostic Features
- **cp**: chest pain type  
  - 1 = typical angina  
  - 2 = atypical angina  
  - 3 = non-anginal pain  
  - 4 = asymptomatic  
- **trestbps**: resting blood pressure (mm Hg on admission to hospital)  
- **chol**: serum cholesterol (mg/dl)  
- **fbs**: fasting blood sugar > 120 mg/dl (1 = true, 0 = false)  
- **restecg**: resting electrocardiographic results  
  - 0 = normal  
  - 1 = ST-T wave abnormality (T wave inversions and/or ST elevation or depression > 0.05 mV)  
  - 2 = probable or definite left ventricular hypertrophy (by Estes’ criteria)  
- **thalach**: maximum heart rate achieved  
- **exang**: exercise-induced angina (1 = yes, 0 = no)  
- **oldpeak**: ST depression induced by exercise relative to rest  
- **slope**: slope of the peak exercise ST segment  
  - 1 = upsloping  
  - 2 = flat  
  - 3 = downsloping  
- **ca**: number of major vessels (0–3) colored by fluoroscopy  
- **thal**: Thallium stress test results  
  - 3 = normal  
  - 6 = fixed defect  
  - 7 = reversible defect  

---

## Target Variable (Class Label)
- **0** → no heart disease  
- **1** → presence of heart disease  


In [26]:
#read the data
df = pd.read_csv("heart_disease.csv")
df.head(16)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,Presence of the diagnosis
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0
5,56,1,2,120,236,0,0,178,0,0.8,1,0.0,3.0,0
6,62,0,4,140,268,0,2,160,0,3.6,3,2.0,3.0,1
7,57,0,4,120,354,0,0,163,1,0.6,1,0.0,3.0,0
8,63,1,4,130,254,0,2,147,0,1.4,2,1.0,7.0,1
9,53,1,4,140,203,1,2,155,1,3.1,3,0.0,7.0,1


In [27]:
#Check missing data
print(df.isna().sum()) # par défaut .sum(0) donc la somme sur les colonnes 
#on cherche combien de cellules n'ont pas de valeurs


age                          0
sex                          0
cp                           0
trestbps                     0
chol                         0
fbs                          0
restecg                      0
thalach                      0
exang                        0
oldpeak                      0
slope                        0
ca                           4
thal                         2
Presence of the diagnosis    0
dtype: int64


In [28]:
#Remove patient with missing data and check again
df = df[df.isna().sum(1)==0] #df.isna() marque les valeurs manquantes et sum(1) compte les valeurs valeurs manquantes sur chaque lignes
#1 veut juste dire axis=1 (axis=1 est la sommes sur le long des lignes et axis:=0 est la somme le long de la colonne)
print(df.isna().sum())

age                          0
sex                          0
cp                           0
trestbps                     0
chol                         0
fbs                          0
restecg                      0
thalach                      0
exang                        0
oldpeak                      0
slope                        0
ca                           0
thal                         0
Presence of the diagnosis    0
dtype: int64


/var/folders/7c/np65yykn28x13ygwf_bqkm140000gn/T/ipykernel_7406/3525711865.py:2: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  df = df[df.isna().sum(1)==0] #df.isna() marque les valeurs manquantes et sum(1) compte les valeurs valeurs manquantes sur chaque lignes


__Question__: How many of the variables in the dataset are continuous, and how many are categorical?

-> Continuous variable (variables that measure a numerical quantity which can take differents values on a scale): age, trestbps, chol, thalach, olpeak (so 5)

Categorical value are group or label: sex, cp,  fbs etc  (so 8)

### Converting variables into numerical values

For now, our variables are a mix of continuous and categorical.

To work with this data, we need to convert these categories into numerical values.

One option is to map each letter to a number between 0 and the total number of categories, using [preprocessing.LabelEncoder](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html)
.

This encoding is not necessarily ideal: for example, an algorithm using Euclidean distance would consider a convex cap (x encoded as 5) to be closer to a sunken cap (s encoded as 4) than to a conical cap (c encoded as 1) — which doesn’t really make sense. However, this is not an issue for tree-based algorithms, which treat categories as such rather than as numeric values. The conversion is only required for implementation purposes.

A [one-hot](http://scikit-learn.org/stable/modules/preprocessing.html) encoding is generally a better choice. Keep in mind, however, that it has the drawback of increasing the number of variables and creating correlated features.

In [29]:
#"your code"
list_of_categorical_variable =['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

df.head(30)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,Presence of the diagnosis
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0
5,56,1,2,120,236,0,0,178,0,0.8,1,0.0,3.0,0
6,62,0,4,140,268,0,2,160,0,3.6,3,2.0,3.0,1
7,57,0,4,120,354,0,0,163,1,0.6,1,0.0,3.0,0
8,63,1,4,130,254,0,2,147,0,1.4,2,1.0,7.0,1
9,53,1,4,140,203,1,2,155,1,3.1,3,0.0,7.0,1


In [ ]:
label_encoder = preprocessing.LabelEncoder() #LabelEncoder crée l’outil d’encodage.
#fit identifie les catégories et leur attribue un entier, 
#puis transform remplace chaque valeur par cet entier.


for col in list_of_categorical_variable:
    df[col] = label_encoder.fit_transform(df[col])


In [25]:
df.head(30)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,Presence of the diagnosis
0,63,1,0,145,233,1,2,150,0,2.3,2,0,1,0
1,67,1,3,160,286,0,2,108,1,1.5,1,3,0,1
2,67,1,3,120,229,0,2,129,1,2.6,1,2,2,1
3,37,1,2,130,250,0,0,187,0,3.5,2,0,0,0
4,41,0,1,130,204,0,2,172,0,1.4,0,0,0,0
5,56,1,1,120,236,0,0,178,0,0.8,0,0,0,0
6,62,0,3,140,268,0,2,160,0,3.6,2,2,0,1
7,57,0,3,120,354,0,0,163,1,0.6,0,0,0,0
8,63,1,3,130,254,0,2,147,0,1.4,1,1,2,1
9,53,1,3,140,203,1,2,155,1,3.1,2,0,2,1


### Creating the X and y data matrices

In [14]:
X = np.array(df.drop(columns=['Presence of the diagnosis']))

In [ ]:
y = np.array(df['Presence of the diagnosis'])
y

In [ ]:
print(X.shape, y.shape)

In [ ]:
len(y[y==1])/len(y)

__Question :__ How many samples (examples) does our dataset contain? How many variables are there? What does the class “1” in $ y $ represent? Are the data balanced?

# 2. Selection and Evaluation

We can now split our data into a training set and a test set, and then define a 10-fold split on the training set to perform cross-validation.

### Training and test sets

In [ ]:
#"Your code: define cross validation with a test set = 30% of the dataset and set the random seed=42"
X_train, X_test, y_train, y_test =

### Cross-validation

In [ ]:
n_folds = 10

# Create a KFold object with n_folds folds and shuffle the data before creating the folds
kf = model_selection.KFold(n_splits=n_folds,
                           shuffle=True # shuffle the data
                          )


# Keep the indices to use the same kfolds mutliple times:
kf_indices = list(kf.split(X_train))

## 3. Decision tree

We will now use a decision tree to train a classifier on our data.

Decision trees are implemented in the [DecisionTreeClassifier](http://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) class from the `tree` module in scikit-learn.

### Decision tree with default hyperparameters

Let’s compute the F1 scores using cross-validation for a decision tree whose hyperparameters are set to the default values in scikit-learn:

In [ ]:
#Define the model
model_tree_default = tree.DecisionTreeClassifier()

In [ ]:
#use the function cross vall
f1_tree_default = model_selection.cross_val_score(model_tree_default, # classifier
                                                  X_train, y_train, # data
                                                  cv=kf_indices, # cross validation indices
                                                  scoring='f1' # metric
                                                  )

In [ ]:
print("Cross validation F1 score : %.3f +/- %.3f" % (np.mean(f1_tree_default), np.std(f1_tree_default)))

__Question :__ what do you think of this performance ?

### Cross-Validation of the Decision Tree Depth

By default (see the documentation), we used a decision tree with maximum depth. We will now treat the tree depth (max_depth) as a hyperparameter to optimize using grid search.

In [ ]:
#Let’s start by defining the grid:
d_values = np.arange(2, 31)

Let's use [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)  :

In [ ]:
# Create a gridSearch object
grid_tree = model_selection.GridSearchCV(tree.DecisionTreeClassifier(), # model
                                         {'max_depth': d_values}, # hyperparameter grids
                                         cv=kf_indices, # cross validation indices
                                         scoring='f1' # metrics
                                         )

In [ ]:
%%time

# Use grid
grid_tree.fit(X_train, y_train)

In [ ]:
#Hyperparameters of the best model are given:
print(grid_tree.best_params_)

Performance of the best model can be retrived using this code:

In [ ]:
mean_test_score = grid_tree.cv_results_['mean_test_score']
stde_test_score = grid_tree.cv_results_['std_test_score'] / np.sqrt(n_folds) # standard error

plt.plot(d_values, mean_test_score)
plt.plot(d_values, (mean_test_score + stde_test_score), '--', color='steelblue')
plt.plot(d_values, (mean_test_score - stde_test_score), '--', color='steelblue')
plt.fill_between(d_values, (mean_test_score + stde_test_score),
                 (mean_test_score - stde_test_score), alpha=0.2)

best_index = np.where(d_values == grid_tree.best_params_['max_depth'])[0][0]
plt.scatter(d_values[best_index], mean_test_score[best_index])

plt.xlabel('Max depth')
plt.ylabel('F1')
plt.title("Cross validation performance as a function of the max depth")

__Question :__ What do you think about this performance ?

### Optimal decision tree

In [ ]:
print("Best cross validated F1 score : %.3f" % grid_tree.best_score_)

Best model:

In [ ]:
model_tree_best = grid_tree.best_estimator_

## 4. Decision tree interpretation

### Visualization

We can visualize the optimal tree using [plot_tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html) from scikit-learn:

In [ ]:
fig = plt.figure(figsize=(25, 20))
tree.plot_tree(model_tree_best, fontsize=12)
plt.show()

__Question :__ Do you think the model is interpretable ?

### Feature importance

To interpret the decision tree, we can also look at the importance of each feature. The higher is a feature’s importance, the more it helps reduce the tree’s classification error.

In [ ]:
fig = plt.figure(figsize=(12, 6))

num_features = X_train.shape[1]

# Plot feature importance
plt.scatter(range(num_features), model_tree_best.feature_importances_,
           label="Decision tree")

#legend
tmp = plt.legend(fontsize=14)

# X axis
plt.xlabel('Features', fontsize=14)
feature_names = list(df.columns[0:-1])
tmp = plt.xticks(range(num_features), feature_names,
                 rotation=90, fontsize=14)

# Y axis
tmp = plt.ylabel('Importance score', fontsize=14)

# Title
tmp = plt.title('Feature importance score', fontsize=16)

### Comparison with logistic regression

We can also compare these feature importances to the regression coefficients of a logistic regression model:

In [ ]:
from sklearn import linear_model

In [ ]:
c_values = np.logspace(-3, 3, 50)

# Data scaling
std_scaler = preprocessing.StandardScaler().fit(X_train)
X_train_scaled = std_scaler.transform(X_train)
X_test_scaled = std_scaler.transform(X_test)

# GridSearch definition
grid_logreg = model_selection.GridSearchCV(linear_model.LogisticRegression(penalty='l2'), # model
                                          {'C': c_values}, # Hyperparameters
                                          cv=kf_indices, # cross-validation indices
                                          scoring='f1' # metric
                                          )

grid_logreg.fit(X_train_scaled, y_train)

In [ ]:
print("Best cross-validated F1 : %.3f" % grid_logreg.best_score_)

__Question :__ Compare the performance of the decision tree and the logistic regression model.

In [ ]:
fig = plt.figure(figsize=(12, 6))

num_features = X_train.shape[1]

# Scale importance scores between 0 and 1
tree_importances = model_tree_best.feature_importances_
tree_importances_min = np.min(tree_importances)
tree_importances_max = np.max(tree_importances)
tree_importances = (tree_importances-tree_importances_min)/(tree_importances_max-tree_importances_min)

# Plot decision tree importance scores
plt.bar(range(num_features), tree_importances,
           label="Decision tree", width=0.4)

# Scale importance scores between 0 and 1
logreg_coeffs = np.abs(grid_logreg.best_estimator_.coef_[0])
logreg_coeffs_min = np.min(logreg_coeffs)
logreg_coeffs_max = np.max(logreg_coeffs)
logreg_coeffs = (logreg_coeffs-logreg_coeffs_min)/(logreg_coeffs_max-logreg_coeffs_min)

# Plot logistic regression importance score
plt.bar((np.arange(num_features)+0.4), logreg_coeffs,
           label="Logistic regression", width=0.4)


# Legend
tmp = plt.legend(fontsize=14)

# X axis
plt.xlabel('Feature', fontsize=14)
feature_names = list(df.columns[:-1])
tmp = plt.xticks(range(num_features), feature_names,
                 rotation=90, fontsize=14)

# Y axis
tmp = plt.ylabel('Importance', fontsize=14)

# Title
tmp = plt.title('Feature importance', fontsize=16)

__Question :__ How do these feature importances compare?

## 5. Random Forest

Can we improve the performance of the decision tree using an ensemble method? Here, we will use a random forest, implemented in the [RandomForestClassifier](http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) from the `ensemble` module in scikit-learn.

### Cross-Validation of the Number of Trees and the Maximum Depth of Each Tree

We will now consider two hyperparameters: the maximum depth of each tree (max_depth) and the number of trees in the forest (n_estimators).

Let's define the grid

In [ ]:

# "your code": define the depth and the number of trees
d_values =
n_values =

#run grid search with the same cv indices than previously

grid_rf =


# Apply GridSearch on the dataset


# print the Optimal hyperparameters :


# plot of model performance as a function of the hyperparameter values (first reorganise scores as a 2D array)


__Question :__ How does the performance of the random forests compare to the previous models?

### Optimal Random Forest

In [ ]:
print("Best cross-validated F1 : %.3f" % grid_rf.best_score_)

Save best model :

In [ ]:
model_rf_best = grid_rf.best_estimator_

### Feature importance

We can once again look at the feature importance for the best random forest model:

In [ ]:
fig = plt.figure(figsize=(12, 6))

num_features = X_train.shape[1]

# Scale scores between 0 and 1
tree_importances = model_tree_best.feature_importances_
tree_importances_min = np.min(tree_importances)
tree_importances_max = np.max(tree_importances)
tree_importances = (tree_importances-tree_importances_min)/(tree_importances_max-tree_importances_min)

# Plot decision tree feature importance scores
plt.bar(range(num_features), tree_importances,
           label="Decision tree", width=0.3)

# Scale scores between 0 and 1
logreg_coeffs = np.abs(grid_logreg.best_estimator_.coef_[0])
logreg_coeffs_min = np.min(logreg_coeffs)
logreg_coeffs_max = np.max(logreg_coeffs)
logreg_coeffs = (logreg_coeffs-logreg_coeffs_min)/(logreg_coeffs_max-logreg_coeffs_min)

# Plot logistic regression feature importance scores
plt.bar((np.arange(num_features)+0.3), logreg_coeffs,
           label="Logistic regression", width=0.3)

# Scale scores between 0 and 1
rf_importances = model_rf_best.feature_importances_
rf_importances_min = np.min(rf_importances)
rf_importances_max = np.max(rf_importances)
rf_importances = (rf_importances-rf_importances_min)/(rf_importances_max-rf_importances_min)

# Plot random forest feature importance scores
plt.bar((np.arange(num_features)+0.6),  rf_importances,
           label="Random Forest", width=0.3)


# Legend
tmp = plt.legend()

# X axis
plt.xlabel('Features')
feature_names = list(df.columns[:-1])
tmp = plt.xticks(range(num_features), feature_names,
                 rotation=90, fontsize=14)

# Y axis
tmp = plt.ylabel('Importance score')

# Titre
tmp = plt.title('Feature importance')

__Question :__ Which variables are now the most important? How does this compare to the previous models?

## 6. Gradient Boosting

Gradient boosting is implemented in scikit-learn in the [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html?highlight=boosting#sklearn.ensemble.GradientBoostingClassifier) from the `ensemble` module.

### Cross-Validation and Hyperparameter Selection

As with random forests, we will optimize here the number of estimators and the depth of the trees.

In [ ]:

# "your code": define the depth and the number of trees
d_values =
n_values =

#run grid search with the same cv indices than previously

grid_boost =


# Apply GridSearch on the dataset


# print the Optimal hyperparameters :


# plot of model performance as a function of the hyperparameter values (first reorganise scores as a 2D array)

__Question :__ How does the performance of gradient boosting change with different hyperparameter values? How does it compare to the previous models’ performance?

### Optimal Gradient Boosting

In [ ]:
print("Best cross-validated F1 : %.3f" % grid_boost.best_score_)

The optimal model is :

In [ ]:
model_boost_best = grid_boost.best_estimator_

### Feature importance

In [ ]:
#"YOUR CODE" plot feature importance for all the previous model

__Question :__ Which variables are now the most important? How does this compare to the previous models?

## 7. Final model

__Question :__ Which of these models would you choose as the best-performing one to classify the mushrooms in the test set?

You will now evaluate the model you have chosen on the test set:

In [ ]:
my_model =  # "YOUR CODE" : write here the name of the model you have chosen.

# Predict on the test set
y_pred = my_model.predict(X_test)

In [ ]:
from sklearn import metrics

In [ ]:
print("F1 score of the chosen model on the test set: %.3f" % metrics.f1_score(y_test, y_pred))

__Question :__ What do you think of this performance? Is there a risk of overfitting?

### Confusion matrix

To better interpret the results, we can also visualize the confusion matrix:

In [ ]:
metrics.ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

Another option is :

In [ ]:
metrics.confusion_matrix(y_test, y_pred)

__Question :__ What do you think of this confusion matrix? Is it satisfactory? Remember, we are trying to predict whether patient has an heart disease.

### ROC curve

We can also evaluate the model’s performance __before thresholding__, that is, using the predicted numerical scores instead of the binary labels, thanks to a [Courbe ROC](https://scikit-learn.org/stable/modules/model_evaluation.html#roc-metrics).

The pre-threshold scores of a scikit-learn classification model can be accessed using the `predict_proba` method.

In [ ]:
y_pred_scores =  my_model.predict_proba(X_test)[:, 1]

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred_scores)
roc_auc = metrics.auc(fpr, tpr)

fig = plt.figure(figsize=(6, 6))

plt.plot(fpr, tpr, lw=2)

# diagonale
plt.plot([0, 1], [0, 1], color='k')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])

plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title("ROC curve of the final model")

We can also use this curve to determine the true positive rate corresponding to a given false positive rate, and to find the corresponding threshold:

In [ ]:
max_fpr = 0.01
max_index_where_fpr_acceptable = np.where(fpr <= max_fpr)[0][-1]
max_tpr = tpr[max_index_where_fpr_acceptable]
print("The true positive rate corresponding to a false positive rate not exceeding %.f %% is %.f %%" % ((100*max_fpr), (100*max_tpr)))
print("It corresponds to a threshold of %.2f on the model’s predictions." % thresholds[max_index_where_fpr_acceptable])